# 19.18 Out-of-distribution detection

OOD detection flags inputs that look unlike the training distribution before trusting ordinary predictions.

We compare a max-softmax shortcut with a real distance score in standardized representation space and evaluate AUROC on held-out ID versus shifted OOD points. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_scaled_logistic(X, y, test_size=0.4, random_state=0):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_te, y_tr, y_te


def fit_scaled_logistic_three_way(X, y, random_state=0):
    if len(y) < 12:
        repeats = int(np.ceil(12 / len(y)))
        X = np.tile(X, (repeats, 1))
        y = np.tile(y, repeats)
    x_tmp, x_te, y_tmp, y_te = train_test_split(X, y, test_size=0.3, random_state=random_state, stratify=y)
    x_tr, x_cal, y_tr, y_cal = train_test_split(x_tmp, y_tmp, test_size=0.35, random_state=random_state, stratify=y_tmp)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_cal = scaler.transform(x_cal)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_cal, x_te, y_tr, y_cal, y_te


def top_class_binary_view(y, proba):
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y).astype(int)
    return correct, conf, pred


def degrade_d5_features(name, X):
    rng = np.random.default_rng(1918)
    if name.startswith("D5"):
        return X + rng.normal(0.0, X.std(axis=0) * 0.35, size=X.shape)
    return X


## The concept, built once

A detector thresholds an unfamiliarity score:

$$\hat o=\mathbf{1}[s(x)>\tau].$$

The lesson OOD scores sum to $4.300$ and the guarded value with knob $1.500$ is $10.750$.

In [ ]:

def ood_detector(train_repr, score_values, tau=None):
    if tau is None:
        tau = float(np.quantile(score_values, 0.95))
    flags = score_values > tau
    return flags.astype(int), tau


def mahalanobis_scores(train_repr, test_repr):
    center = train_repr.mean(axis=0)
    variances = train_repr.var(axis=0) + 1e-6
    centered = test_repr - center
    return np.sqrt(((centered ** 2) / variances).sum(axis=1))

scores = np.array([0.800, 1.400, 2.100])
total = float(scores.sum())
abs_mass = float(np.abs(scores).sum())
share = float(abs(scores[0]) / abs_mass)
guarded = total + 1.5 * abs_mass
contrast = total - scores[2]
change = contrast - total
flags, tau = ood_detector(np.zeros((3, 2)), scores, tau=1.5)

assert np.isclose(total, 4.300)
assert np.isclose(share, 0.186, atol=0.001)
assert np.isclose(guarded, 10.750)
assert np.isclose(change, -2.100)
print("total", total, "guard", guarded, "flags", flags.tolist())


The reusable rung method fits on ID train data, creates shifted OOD test points without downloads, scores both ID and OOD with a Mahalanobis-style distance, and reports AUROC.

In [ ]:

def rung_ood(X, y, shift_strength=2.5):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    rng = np.random.default_rng(1918)
    x_ood = x_te + rng.normal(shift_strength, 0.5, size=x_te.shape)
    id_scores = mahalanobis_scores(x_tr, x_te)
    ood_scores = mahalanobis_scores(x_tr, x_ood)
    labels = np.r_[np.zeros(len(id_scores)), np.ones(len(ood_scores))]
    scores = np.r_[id_scores, ood_scores]
    auroc = roc_auc_score(labels, scores)
    tau = float(np.quantile(id_scores, 0.95))
    fpr = float(np.mean(id_scores > tau))
    recall = float(np.mean(ood_scores > tau))
    return float(auroc), fpr, recall, id_scores, ood_scores, tau


## The dataset ladder

All six notebooks in this batch use the same F15 classification ladder: a hand-checkable D1 toy, synthetic rungs, two real tabular rungs, and the real D5 Breast Cancer stress rung. The method changes, not the data-loading story.

In [ ]:

rungs = clf_ladder()

for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, : min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  classes:", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


In [ ]:

rows = []
for rung, (name, X, y) in enumerate(rungs, start=1):
    auroc, fpr, recall, id_scores, ood_scores, tau = rung_ood(X, y)
    rows.append((rung, name, auroc, fpr, recall))

print("rung | AUROC | FPR@ID95 | recall")
for rung, name, auroc, fpr, recall in rows:
    print(f"D{rung} | {auroc:.3f} | {fpr:.3f} | {recall:.3f} | {name}")


In [ ]:

summary = []
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for col, (name, X, y) in enumerate(rungs):
    auroc, fpr, recall, id_scores, ood_scores, tau = rung_ood(X, y)
    summary.append(auroc)
    axes[col].hist(id_scores, alpha=0.6, label="ID")
    axes[col].hist(ood_scores, alpha=0.6, label="OOD")
    axes[col].axvline(tau, color="black", linestyle="--")
    axes[col].set_title(name.split("(")[0].strip(), fontsize=9)
    axes[col].set_xlabel("distance score")
    axes[col].legend(fontsize=7)

plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), summary, marker="o")
plt.xlabel("ladder rung")
plt.ylabel("AUROC")
plt.title("OOD AUROC vs. D1→D5")
plt.xticks(range(1, 6))
plt.ylim(0, 1.05)
plt.show()


## Pitfall on D5: using maximum softmax alone

Max softmax can be high far from the training distribution. The fix scores representation distance and chooses $	au$ from ID calibration data, not future OOD examples.

In [ ]:

name, X, y = rungs[-1]
clf, x_tr, x_te, y_tr, y_te = fit_scaled_logistic(X, y)
rng = np.random.default_rng(1918)
x_ood = x_te + rng.normal(2.5, 0.5, size=x_te.shape)
id_conf = clf.predict_proba(x_te).max(axis=1)
ood_conf = clf.predict_proba(x_ood).max(axis=1)
softmax_labels = np.r_[np.zeros(len(id_conf)), np.ones(len(ood_conf))]
softmax_scores = -np.r_[id_conf, ood_conf]
softmax_auroc = roc_auc_score(softmax_labels, softmax_scores)
distance_auroc, fpr, recall, id_scores, ood_scores, tau = rung_ood(X, y)

print("max-softmax AUROC", softmax_auroc)
print("distance AUROC", distance_auroc)
print("tau from ID scores", tau)
print("recall at ID-selected tau", recall)


## Evaluate it + Practice

- Metric: OOD detection AUROC and FPR at target recall.
- No-skill baseline: random scores, expected AUROC near 0.5.
- Cheap sanity check: ID scores should mostly fall below the ID-selected tau.
- Ablation: replace distance with negative max softmax and compare AUROC.
- Failure signals: tau tuned on future OOD, high-confidence far-away points, or shift types mixed together.

Practice 1: Change the random seed or stress strength and explain which rung moves most.

Practice 2: Lower shift strength and observe AUROC degrade.

Practice 3: Compare covariate shift to label permutation as separate stress cases.